In [1]:
# ============================================
# WEEK 5 BBO INPUT GENERATION
# Advanced neural-network surrogate approach
# ============================================

import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.utils import resample

np.random.seed(42)

# =========================================================
# HISTORICAL INPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_inputs = [
    np.array([0.211325, 0.788675]),
    np.array([0.723607, 0.276393]),
    np.array([0.166667, 0.500000, 0.833333]),
    np.array([0.125000, 0.375000, 0.625000, 0.875000]),
    np.array([0.875000, 0.625000, 0.375000, 0.125000]),
    np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
    np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
    np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
]

week2_inputs = [
    np.array([0.654299, 0.353479]),
    np.array([0.754299, 0.253479]),
    np.array([0.304299, 0.553479, 0.709691]),
    np.array([0.804299, 0.603479, 0.409691, 0.205016]),
    np.array([0.854299, 0.653479, 0.359691, 0.155016]),
    np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
    np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
    np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
]

week3_inputs = [
    np.array([0.582812, 0.421077]),
    np.array([0.712865, 0.284413]),
    np.array([0.118496, 0.481282, 0.876608]),
    np.array([1.000000, 0.683447, 0.334333, 0.000000]),
    np.array([0.882245, 0.615032, 0.380358, 0.114494]),
    np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
    np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
    np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
]

week4_inputs = [
    np.array([0.183704, 0.127166]),
    np.array([0.255353, 0.749841]),
    np.array([0.094906, 0.770362, 0.859589]),
    np.array([0.820856, 0.226975, 0.784625, 0.113609]),
    np.array([0.917860, 0.273128, 0.475575, 0.052446]),
    np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
    np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
    np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
]

# =========================================================
# HISTORICAL OUTPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_outputs = np.array([
    1.1327056288856873e-125,
    0.5675786892822564,
    -0.032385408076210126,
    -17.20744048260943,
    60.223125,
    -1.3287857969718009,
    0.34356041660314957,
    9.0501517903694
])

week2_outputs = np.array([
    1.1867858443665185e-41,
    0.2715258567299176,
    -0.1198581070659559,
    -13.082213230390916,
    50.179390256321376,
    -1.5080002951000964,
    0.31639485635485903,
    9.0219949134074
])

week3_outputs = np.array([
    7.99676981308551e-19,
    0.5213723465552891,
    -0.04726977098841539,
    -25.67625344929702,
    64.78245026474816,
    -1.7372122723701597,
    0.3507828450585503,
    9.0587238074074
])

week4_outputs = np.array([
    -2.410121917336285e-100,
    0.0244939290195959,
    -0.04207093453964322,
    -20.330739763644917,
    61.278397876784794,
    -2.4624737429462145,
    0.0634803557047261,
    8.275283250689899
])

# =========================================================
# HELPER FUNCTIONS
# =========================================================
def build_history_for_function(fn_idx):
    X = np.vstack([
        week1_inputs[fn_idx],
        week2_inputs[fn_idx],
        week3_inputs[fn_idx],
        week4_inputs[fn_idx],
    ])
    y = np.array([
        week1_outputs[fn_idx],
        week2_outputs[fn_idx],
        week3_outputs[fn_idx],
        week4_outputs[fn_idx],
    ])
    return X, y

def augment_local_data(X, y, noise_std=0.03, copies=6):
    """
    Light augmentation around observed points.
    This is not true new data; it stabilizes the neural net.
    """
    X_aug = [X]
    y_aug = [y]

    for _ in range(copies):
        noise = np.random.normal(0, noise_std, size=X.shape)
        Xn = np.clip(X + noise, 0, 1)
        yn = y + np.random.normal(0, max(1e-4, np.std(y) * 0.03), size=y.shape)
        X_aug.append(Xn)
        y_aug.append(yn)

    return np.vstack(X_aug), np.concatenate(y_aug)

def make_ensemble_model(random_state):
    """
    Small but deeper MLP than before.
    Using early stopping and regularization helps with tiny data.
    """
    model = Pipeline([
        ("x_scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            hidden_layer_sizes=(32, 16),
            activation="relu",
            solver="adam",
            alpha=1e-3,                 # L2 regularization
            learning_rate_init=0.01,
            max_iter=3000,
            early_stopping=True,
            validation_fraction=0.2,
            n_iter_no_change=40,
            random_state=random_state
        ))
    ])
    return model

def fit_ensemble(X, y, n_models=15):
    """
    Bootstrap ensemble for mean prediction + uncertainty estimate.
    """
    y_scaler = StandardScaler()
    y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

    models = []
    for seed in range(n_models):
        Xb, yb = resample(X, y_scaled, replace=True, random_state=100 + seed)

        # if bootstrap collapses too much, just reuse whole set
        if len(np.unique(yb)) < 2:
            Xb, yb = X, y_scaled

        model = make_ensemble_model(random_state=200 + seed)
        model.fit(Xb, yb)
        models.append(model)

    return models, y_scaler

def predict_ensemble(models, y_scaler, Xcand):
    preds = []
    for m in models:
        p = m.predict(Xcand)
        preds.append(p)
    preds = np.array(preds)  # [n_models, n_candidates]

    mean_scaled = preds.mean(axis=0)
    std_scaled = preds.std(axis=0)

    mean = y_scaler.inverse_transform(mean_scaled.reshape(-1, 1)).ravel()
    std = std_scaled * y_scaler.scale_[0]
    return mean, std

def generate_candidates(X_hist, y_hist, n_local=5000, n_global=1500):
    dim = X_hist.shape[1]

    # rank observed points
    order = np.argsort(y_hist)[::-1]
    x_best = X_hist[order[0]]
    x_second = X_hist[order[1]]

    # trust-region around best and second-best
    local1 = np.clip(x_best + np.random.normal(0, 0.08, size=(n_local // 2, dim)), 0, 1)
    local2 = np.clip(x_second + np.random.normal(0, 0.12, size=(n_local // 2, dim)), 0, 1)

    # interpolation candidates between top regions
    mix = np.random.uniform(0, 1, size=(1000, 1))
    interp = np.clip(mix * x_best + (1 - mix) * x_second + np.random.normal(0, 0.03, size=(1000, dim)), 0, 1)

    # global exploration
    global_cand = np.random.uniform(0, 1, size=(n_global, dim))

    # historical points too
    all_cand = np.vstack([local1, local2, interp, global_cand, X_hist])
    return all_cand, x_best

def acquisition_score(mean, std, Xcand, x_best, exploit_weight=1.0, explore_weight=0.35, trust_penalty=0.10):
    """
    Advanced scoring:
    - mean prediction: exploit good regions
    - uncertainty bonus: explore where ensemble disagrees
    - trust-region penalty: avoid very wild jumps
    """
    dist = np.linalg.norm(Xcand - x_best, axis=1)
    score = exploit_weight * mean + explore_weight * std - trust_penalty * dist
    return score

def propose_week5_for_function(fn_idx):
    X_hist, y_hist = build_history_for_function(fn_idx)

    # augment data lightly
    X_train, y_train = augment_local_data(X_hist, y_hist, noise_std=0.025, copies=8)

    # fit deep-ish ensemble surrogate
    models, y_scaler = fit_ensemble(X_train, y_train, n_models=15)

    # candidate generation
    Xcand, x_best = generate_candidates(X_hist, y_hist)

    # ensemble prediction
    mean, std = predict_ensemble(models, y_scaler, Xcand)

    # acquisition score
    score = acquisition_score(mean, std, Xcand, x_best,
                              exploit_weight=1.0,
                              explore_weight=0.30,
                              trust_penalty=0.08)

    # choose best candidate
    best_idx = np.argmax(score)
    x5 = np.clip(Xcand[best_idx], 0, 1)

    return np.round(x5, 6), X_hist, y_hist, mean[best_idx], std[best_idx]

# =========================================================
# GENERATE WEEK 5 INPUTS
# =========================================================
week5_inputs = []

print("Suggested Week 5 inputs (arrays):")
for fn_idx in range(8):
    x5, X_hist, y_hist, pred_mean, pred_std = propose_week5_for_function(fn_idx)
    week5_inputs.append(x5)
    print(f"Function {fn_idx+1}: {x5} | predicted mean={pred_mean:.6f}, uncertainty={pred_std:.6f}")

print("\nPortal-ready format:")
for i, arr in enumerate(week5_inputs, start=1):
    formatted = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {formatted}")

Suggested Week 5 inputs (arrays):
Function 1: [0.582812 0.421077] | predicted mean=0.000002, uncertainty=0.000023
Function 2: [0.233302 0.003008] | predicted mean=0.616405, uncertainty=0.450684
Function 3: [0.166667 0.5      0.833333] | predicted mean=-0.035607, uncertainty=0.004755
Function 4: [0.080067 0.998896 0.010531 0.591247] | predicted mean=-9.117338, uncertainty=3.230946
Function 5: [0.904371 0.456577 0.033012 0.      ] | predicted mean=73.675095, uncertainty=14.565932
Function 6: [0.981388 0.858165 0.944426 0.435848 0.993187] | predicted mean=-0.801115, uncertainty=0.572097
Function 7: [1.       0.062448 0.87468  0.013646 0.513122 1.      ] | predicted mean=0.436095, uncertainty=0.055274
Function 8: [0.       0.125529 0.065922 0.24671  0.536416 0.674069 0.964448 0.958795] | predicted mean=9.256215, uncertainty=0.138228

Portal-ready format:
Function 1: 0.582812-0.421077
Function 2: 0.233302-0.003008
Function 3: 0.166667-0.500000-0.833333
Function 4: 0.080067-0.998896-0.010531

In [2]:
# ============================================
# WEEK 5 BBO INPUT GENERATION - PYTORCH VERSION
# Advanced surrogate with backpropagation + Adam
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)

# =========================================================
# HISTORICAL INPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_inputs = [
    np.array([0.211325, 0.788675]),
    np.array([0.723607, 0.276393]),
    np.array([0.166667, 0.500000, 0.833333]),
    np.array([0.125000, 0.375000, 0.625000, 0.875000]),
    np.array([0.875000, 0.625000, 0.375000, 0.125000]),
    np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
    np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
    np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
]

week2_inputs = [
    np.array([0.654299, 0.353479]),
    np.array([0.754299, 0.253479]),
    np.array([0.304299, 0.553479, 0.709691]),
    np.array([0.804299, 0.603479, 0.409691, 0.205016]),
    np.array([0.854299, 0.653479, 0.359691, 0.155016]),
    np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
    np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
    np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
]

week3_inputs = [
    np.array([0.582812, 0.421077]),
    np.array([0.712865, 0.284413]),
    np.array([0.118496, 0.481282, 0.876608]),
    np.array([1.000000, 0.683447, 0.334333, 0.000000]),
    np.array([0.882245, 0.615032, 0.380358, 0.114494]),
    np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
    np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
    np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
]

week4_inputs = [
    np.array([0.183704, 0.127166]),
    np.array([0.255353, 0.749841]),
    np.array([0.094906, 0.770362, 0.859589]),
    np.array([0.820856, 0.226975, 0.784625, 0.113609]),
    np.array([0.917860, 0.273128, 0.475575, 0.052446]),
    np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
    np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
    np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
]

# =========================================================
# HISTORICAL OUTPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_outputs = np.array([
    1.1327056288856873e-125,
    0.5675786892822564,
    -0.032385408076210126,
    -17.20744048260943,
    60.223125,
    -1.3287857969718009,
    0.34356041660314957,
    9.0501517903694
])

week2_outputs = np.array([
    1.1867858443665185e-41,
    0.2715258567299176,
    -0.1198581070659559,
    -13.082213230390916,
    50.179390256321376,
    -1.5080002951000964,
    0.31639485635485903,
    9.0219949134074
])

week3_outputs = np.array([
    7.99676981308551e-19,
    0.5213723465552891,
    -0.04726977098841539,
    -25.67625344929702,
    64.78245026474816,
    -1.7372122723701597,
    0.3507828450585503,
    9.0587238074074
])

week4_outputs = np.array([
    -2.410121917336285e-100,
    0.0244939290195959,
    -0.04207093453964322,
    -20.330739763644917,
    61.278397876784794,
    -2.4624737429462145,
    0.0634803557047261,
    8.275283250689899
])

# =========================================================
# HELPERS
# =========================================================
def build_history_for_function(fn_idx):
    X = np.vstack([
        week1_inputs[fn_idx],
        week2_inputs[fn_idx],
        week3_inputs[fn_idx],
        week4_inputs[fn_idx],
    ]).astype(np.float32)

    y = np.array([
        week1_outputs[fn_idx],
        week2_outputs[fn_idx],
        week3_outputs[fn_idx],
        week4_outputs[fn_idx],
    ], dtype=np.float32)
    return X, y

def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0, keepdims=True)
    if std is None:
        std = X.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return (X - mean) / std, mean, std

def augment_data(X, y, copies=10, x_noise=0.025, y_noise_scale=0.03):
    X_list = [X]
    y_list = [y]
    y_std = max(np.std(y), 1e-4)

    for _ in range(copies):
        Xn = np.clip(X + np.random.normal(0, x_noise, size=X.shape), 0, 1)
        yn = y + np.random.normal(0, y_std * y_noise_scale, size=y.shape)
        X_list.append(Xn)
        y_list.append(yn)

    return np.vstack(X_list), np.concatenate(y_list)

class SurrogateNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_single_model(X, y, seed=0, epochs=1200, lr=0.01, weight_decay=1e-4):
    torch.manual_seed(seed)

    # scale X and y
    Xs, x_mean, x_std = standardize(X)
    ys, y_mean, y_std = standardize(y.reshape(-1, 1))
    ys = ys.ravel()

    X_tensor = torch.tensor(Xs, dtype=torch.float32)
    y_tensor = torch.tensor(ys.reshape(-1, 1), dtype=torch.float32)

    model = SurrogateNet(input_dim=X.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_loss = float("inf")
    patience = 120
    patience_count = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        preds = model(X_tensor)
        loss = loss_fn(preds, y_tensor)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        if current_loss < best_loss - 1e-6:
            best_loss = current_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, x_mean, x_std, y_mean.item(), y_std.item()

def fit_ensemble(X, y, n_models=20):
    models = []
    for i in range(n_models):
        Xb_idx = np.random.choice(len(X), size=len(X), replace=True)
        Xb = X[Xb_idx]
        yb = y[Xb_idx]
        model_info = train_single_model(Xb, yb, seed=100 + i)
        models.append(model_info)
    return models

def ensemble_predict(models, Xcand):
    preds = []

    for model, x_mean, x_std, y_mean, y_std in models:
        Xs = (Xcand - x_mean) / x_std
        X_tensor = torch.tensor(Xs, dtype=torch.float32)

        model.eval()
        with torch.no_grad():
            p = model(X_tensor).cpu().numpy().ravel()

        # inverse transform y
        p = p * y_std + y_mean
        preds.append(p)

    preds = np.array(preds)
    return preds.mean(axis=0), preds.std(axis=0)

def generate_candidates(X_hist, y_hist, n_local=5000, n_global=1500):
    order = np.argsort(y_hist)[::-1]
    x_best = X_hist[order[0]]
    x_second = X_hist[order[1]]
    dim = X_hist.shape[1]

    local1 = np.clip(x_best + np.random.normal(0, 0.07, size=(n_local // 2, dim)), 0, 1)
    local2 = np.clip(x_second + np.random.normal(0, 0.11, size=(n_local // 2, dim)), 0, 1)

    mix = np.random.uniform(0, 1, size=(1000, 1))
    interp = np.clip(
        mix * x_best + (1 - mix) * x_second + np.random.normal(0, 0.025, size=(1000, dim)),
        0, 1
    )

    global_cand = np.random.uniform(0, 1, size=(n_global, dim))
    Xcand = np.vstack([local1, local2, interp, global_cand, X_hist])

    return Xcand, x_best

def acquisition(mean, std, Xcand, x_best, explore_weight=0.25, distance_penalty=0.06):
    dist = np.linalg.norm(Xcand - x_best, axis=1)
    return mean + explore_weight * std - distance_penalty * dist

def propose_week5(fn_idx):
    X_hist, y_hist = build_history_for_function(fn_idx)

    # light augmentation to stabilize deep surrogate on tiny data
    X_train, y_train = augment_data(X_hist, y_hist, copies=12)

    models = fit_ensemble(X_train, y_train, n_models=20)

    Xcand, x_best = generate_candidates(X_hist, y_hist)
    mean, std = ensemble_predict(models, Xcand)

    score = acquisition(mean, std, Xcand, x_best,
                        explore_weight=0.25,
                        distance_penalty=0.06)

    best_idx = np.argmax(score)
    x5 = np.clip(Xcand[best_idx], 0, 1)
    return np.round(x5, 6), mean[best_idx], std[best_idx]

# =========================================================
# RUN FOR ALL 8 FUNCTIONS
# =========================================================
week5_inputs = []

print("Suggested Week 5 inputs (arrays):")
for fn_idx in range(8):
    x5, pred_mean, pred_std = propose_week5(fn_idx)
    week5_inputs.append(x5)
    print(f"Function {fn_idx+1}: {x5} | predicted mean={pred_mean:.6f} | uncertainty={pred_std:.6f}")

print("\nPortal-ready format:")
for i, arr in enumerate(week5_inputs, start=1):
    portal_string = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {portal_string}")

ModuleNotFoundError: No module named 'torch'

In [3]:
!pip install torch torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 28.1 MB/s  0:00:028.7 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 26.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.9/679.9 kB 18.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [torchvision]━━━━━━ 2/3 [torchvision]


In [4]:
# ============================================
# WEEK 5 BBO INPUT GENERATION - PYTORCH VERSION
# Advanced surrogate with backpropagation + Adam
# ============================================

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

np.random.seed(42)
torch.manual_seed(42)

# =========================================================
# HISTORICAL INPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_inputs = [
    np.array([0.211325, 0.788675]),
    np.array([0.723607, 0.276393]),
    np.array([0.166667, 0.500000, 0.833333]),
    np.array([0.125000, 0.375000, 0.625000, 0.875000]),
    np.array([0.875000, 0.625000, 0.375000, 0.125000]),
    np.array([0.150000, 0.350000, 0.550000, 0.750000, 0.950000]),
    np.array([0.900000, 0.100000, 0.700000, 0.300000, 0.500000, 0.800000]),
    np.array([0.111111, 0.222222, 0.333333, 0.444444, 0.555555, 0.666667, 0.777778, 0.888889])
]

week2_inputs = [
    np.array([0.654299, 0.353479]),
    np.array([0.754299, 0.253479]),
    np.array([0.304299, 0.553479, 0.709691]),
    np.array([0.804299, 0.603479, 0.409691, 0.205016]),
    np.array([0.854299, 0.653479, 0.359691, 0.155016]),
    np.array([0.904299, 0.703479, 0.509691, 0.305016, 0.103275]),
    np.array([0.884299, 0.123479, 0.729691, 0.285016, 0.523275, 0.787492]),
    np.array([0.104299, 0.243479, 0.319691, 0.465016, 0.573275, 0.647492, 0.794889, 0.860861])
]

week3_inputs = [
    np.array([0.582812, 0.421077]),
    np.array([0.712865, 0.284413]),
    np.array([0.118496, 0.481282, 0.876608]),
    np.array([1.000000, 0.683447, 0.334333, 0.000000]),
    np.array([0.882245, 0.615032, 0.380358, 0.114494]),
    np.array([0.000000, 0.226282, 0.564108, 0.905744, 1.000000]),
    np.array([0.905495, 0.091782, 0.689608, 0.305244, 0.491854, 0.804378]),
    np.array([0.113495, 0.214782, 0.338108, 0.437244, 0.549353, 0.673378, 0.771789, 0.898699])
]

week4_inputs = [
    np.array([0.183704, 0.127166]),
    np.array([0.255353, 0.749841]),
    np.array([0.094906, 0.770362, 0.859589]),
    np.array([0.820856, 0.226975, 0.784625, 0.113609]),
    np.array([0.917860, 0.273128, 0.475575, 0.052446]),
    np.array([0.052509, 0.789636, 0.354234, 0.228337, 0.889991]),
    np.array([0.833796, 0.096195, 0.232612, 0.789158, 0.185769, 0.681185]),
    np.array([0.448564, 0.335191, 0.751694, 0.277261, 0.167614, 0.717472, 0.304117, 0.767059])
]

# =========================================================
# HISTORICAL OUTPUTS: WEEK 1, 2, 3, 4
# =========================================================
week1_outputs = np.array([
    1.1327056288856873e-125,
    0.5675786892822564,
    -0.032385408076210126,
    -17.20744048260943,
    60.223125,
    -1.3287857969718009,
    0.34356041660314957,
    9.0501517903694
])

week2_outputs = np.array([
    1.1867858443665185e-41,
    0.2715258567299176,
    -0.1198581070659559,
    -13.082213230390916,
    50.179390256321376,
    -1.5080002951000964,
    0.31639485635485903,
    9.0219949134074
])

week3_outputs = np.array([
    7.99676981308551e-19,
    0.5213723465552891,
    -0.04726977098841539,
    -25.67625344929702,
    64.78245026474816,
    -1.7372122723701597,
    0.3507828450585503,
    9.0587238074074
])

week4_outputs = np.array([
    -2.410121917336285e-100,
    0.0244939290195959,
    -0.04207093453964322,
    -20.330739763644917,
    61.278397876784794,
    -2.4624737429462145,
    0.0634803557047261,
    8.275283250689899
])

# =========================================================
# HELPERS
# =========================================================
def build_history_for_function(fn_idx):
    X = np.vstack([
        week1_inputs[fn_idx],
        week2_inputs[fn_idx],
        week3_inputs[fn_idx],
        week4_inputs[fn_idx],
    ]).astype(np.float32)

    y = np.array([
        week1_outputs[fn_idx],
        week2_outputs[fn_idx],
        week3_outputs[fn_idx],
        week4_outputs[fn_idx],
    ], dtype=np.float32)
    return X, y

def standardize(X, mean=None, std=None):
    if mean is None:
        mean = X.mean(axis=0, keepdims=True)
    if std is None:
        std = X.std(axis=0, keepdims=True)
    std = np.where(std < 1e-8, 1.0, std)
    return (X - mean) / std, mean, std

def augment_data(X, y, copies=10, x_noise=0.025, y_noise_scale=0.03):
    X_list = [X]
    y_list = [y]
    y_std = max(np.std(y), 1e-4)

    for _ in range(copies):
        Xn = np.clip(X + np.random.normal(0, x_noise, size=X.shape), 0, 1)
        yn = y + np.random.normal(0, y_std * y_noise_scale, size=y.shape)
        X_list.append(Xn)
        y_list.append(yn)

    return np.vstack(X_list), np.concatenate(y_list)

class SurrogateNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Dropout(0.10),
            nn.Linear(16, 1)
        )

    def forward(self, x):
        return self.net(x)

def train_single_model(X, y, seed=0, epochs=1200, lr=0.01, weight_decay=1e-4):
    torch.manual_seed(seed)

    # scale X and y
    Xs, x_mean, x_std = standardize(X)
    ys, y_mean, y_std = standardize(y.reshape(-1, 1))
    ys = ys.ravel()

    X_tensor = torch.tensor(Xs, dtype=torch.float32)
    y_tensor = torch.tensor(ys.reshape(-1, 1), dtype=torch.float32)

    model = SurrogateNet(input_dim=X.shape[1])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    best_state = None
    best_loss = float("inf")
    patience = 120
    patience_count = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        preds = model(X_tensor)
        loss = loss_fn(preds, y_tensor)
        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        if current_loss < best_loss - 1e-6:
            best_loss = current_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        if patience_count >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, x_mean, x_std, y_mean.item(), y_std.item()

def fit_ensemble(X, y, n_models=20):
    models = []
    for i in range(n_models):
        Xb_idx = np.random.choice(len(X), size=len(X), replace=True)
        Xb = X[Xb_idx]
        yb = y[Xb_idx]
        model_info = train_single_model(Xb, yb, seed=100 + i)
        models.append(model_info)
    return models

def ensemble_predict(models, Xcand):
    preds = []

    for model, x_mean, x_std, y_mean, y_std in models:
        Xs = (Xcand - x_mean) / x_std
        X_tensor = torch.tensor(Xs, dtype=torch.float32)

        model.eval()
        with torch.no_grad():
            p = model(X_tensor).cpu().numpy().ravel()

        # inverse transform y
        p = p * y_std + y_mean
        preds.append(p)

    preds = np.array(preds)
    return preds.mean(axis=0), preds.std(axis=0)

def generate_candidates(X_hist, y_hist, n_local=5000, n_global=1500):
    order = np.argsort(y_hist)[::-1]
    x_best = X_hist[order[0]]
    x_second = X_hist[order[1]]
    dim = X_hist.shape[1]

    local1 = np.clip(x_best + np.random.normal(0, 0.07, size=(n_local // 2, dim)), 0, 1)
    local2 = np.clip(x_second + np.random.normal(0, 0.11, size=(n_local // 2, dim)), 0, 1)

    mix = np.random.uniform(0, 1, size=(1000, 1))
    interp = np.clip(
        mix * x_best + (1 - mix) * x_second + np.random.normal(0, 0.025, size=(1000, dim)),
        0, 1
    )

    global_cand = np.random.uniform(0, 1, size=(n_global, dim))
    Xcand = np.vstack([local1, local2, interp, global_cand, X_hist])

    return Xcand, x_best

def acquisition(mean, std, Xcand, x_best, explore_weight=0.25, distance_penalty=0.06):
    dist = np.linalg.norm(Xcand - x_best, axis=1)
    return mean + explore_weight * std - distance_penalty * dist

def propose_week5(fn_idx):
    X_hist, y_hist = build_history_for_function(fn_idx)

    # light augmentation to stabilize deep surrogate on tiny data
    X_train, y_train = augment_data(X_hist, y_hist, copies=12)

    models = fit_ensemble(X_train, y_train, n_models=20)

    Xcand, x_best = generate_candidates(X_hist, y_hist)
    mean, std = ensemble_predict(models, Xcand)

    score = acquisition(mean, std, Xcand, x_best,
                        explore_weight=0.25,
                        distance_penalty=0.06)

    best_idx = np.argmax(score)
    x5 = np.clip(Xcand[best_idx], 0, 1)
    return np.round(x5, 6), mean[best_idx], std[best_idx]

# =========================================================
# RUN FOR ALL 8 FUNCTIONS
# =========================================================
week5_inputs = []

print("Suggested Week 5 inputs (arrays):")
for fn_idx in range(8):
    x5, pred_mean, pred_std = propose_week5(fn_idx)
    week5_inputs.append(x5)
    print(f"Function {fn_idx+1}: {x5} | predicted mean={pred_mean:.6f} | uncertainty={pred_std:.6f}")

print("\nPortal-ready format:")
for i, arr in enumerate(week5_inputs, start=1):
    portal_string = "-".join(f"{x:.6f}" for x in arr)
    print(f"Function {i}: {portal_string}")

Suggested Week 5 inputs (arrays):
Function 1: [0.582812 0.421077] | predicted mean=0.000001 | uncertainty=0.000001
Function 2: [0.684812 0.281383] | predicted mean=0.531801 | uncertainty=0.024871
Function 3: [0.171685 0.508728 0.819757] | predicted mean=-0.033087 | uncertainty=0.001145
Function 4: [0.734042 0.687625 0.466427 0.303264] | predicted mean=-12.693433 | uncertainty=0.660961
Function 5: [0.568707 0.909615 0.957143 0.01142 ] | predicted mean=87.186028 | uncertainty=28.500397
Function 6: [0.744968 0.084644 0.801527 0.169728 0.990511] | predicted mean=-1.014570 | uncertainty=0.281574
Function 7: [1.       0.229371 0.690054 0.244152 0.521424 1.      ] | predicted mean=0.413583 | uncertainty=0.065466
Function 8: [0.02262  0.       0.288709 0.762817 0.551656 0.961269 0.843087 1.      ] | predicted mean=9.173775 | uncertainty=0.127390

Portal-ready format:
Function 1: 0.582812-0.421077
Function 2: 0.684812-0.281383
Function 3: 0.171685-0.508728-0.819757
Function 4: 0.734042-0.687625